In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_path = "/content/drive/MyDrive/Cow dataset/train"
test_path  = "/content/drive/MyDrive/Cow dataset/test"

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    zoom_range=0.4,
    shear_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.6,1.4],
    fill_mode='nearest'
)

In [ ]:
train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

# Define test_gen for the test data, typically with just rescaling
test_gen = ImageDataGenerator(
    rescale=1./255
)

test_data = test_gen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 1445 images belonging to 4 classes.
Found 108 images belonging to 4 classes.


In [ ]:
class_names = list(train_data.class_indices.keys())
print("Class order:", class_names)

Class order: ['Deoni', 'Gir', 'Jersey_Crossbred', 'Khillari']


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
for layer in base_model.layers[:-50]:
    layer.trainable = False
for layer in base_model.layers[-50:]:
    layer.trainable = True

In [ ]:
from tensorflow.keras.layers import BatchNormalization

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)

output = Dense(4, activation='softmax')(x)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model # Import Model class

# Instantiate the model_cow by connecting base_model's input to the output layer
model_cow = Model(inputs=base_model.input, outputs=output)

model_cow.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stop = EarlyStopping(patience=5, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/best_cow_model.h5",
    monitor='val_accuracy',
    save_best_only=True
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stop = EarlyStopping(patience=5, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/best_cow_model.h5",
    monitor='val_accuracy',
    save_best_only=True
)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(weights))

print(class_weights)

{0: np.float64(4.881756756756757), 1: np.float64(0.34437559580552907), 2: np.float64(2.675925925925926), 3: np.float64(1.9318181818181819)}


In [ ]:
history = model_cow.fit(
    train_data,
    validation_data=test_data,
    epochs=30,
    callbacks=[early_stop, reduce_lr, checkpoint],
    class_weight=class_weights   # ✔ correct name
)

Epoch 1/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.3317 - loss: 2.1182 

46/46 ━━━━━━━━━━━━━━━━━━━━ 641s 13s/step - accuracy: 0.3474 - loss: 1.9399 - val_accuracy: 0.5093 - val_loss: 1.0976 - learning_rate: 1.0000e-04
Epoch 2/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 41s 889ms/step - accuracy: 0.4394 - loss: 1.2881 - val_accuracy: 0.4444 - val_loss: 1.0740 - learning_rate: 1.0000e-04
Epoch 3/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 876ms/step - accuracy: 0.5433 - loss: 1.1122 - val_accuracy: 0.4815 - val_loss: 1.0657 - learning_rate: 1.0000e-04
Epoch 4/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 810ms/step - accuracy: 0.5856 - loss: 0.9184

46/46 ━━━━━━━━━━━━━━━━━━━━ 41s 887ms/step - accuracy: 0.5834 - loss: 0.9802 - val_accuracy: 0.5556 - val_loss: 0.9575 - learning_rate: 1.0000e-04
Epoch 5/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 814ms/step - accuracy: 0.6091 - loss: 0.9806

46/46 ━━━━━━━━━━━━━━━━━━━━ 42s 903ms/step - accuracy: 0.6304 - loss: 0.8579 - val_accuracy: 0.6204 - val_loss: 0.9797 - learning_rate: 1.0000e-04
Epoch 6/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 831ms/step - accuracy: 0.6954 - loss: 0.7394

46/46 ━━━━━━━━━━━━━━━━━━━━ 42s 912ms/step - accuracy: 0.6872 - loss: 0.7163 - val_accuracy: 0.6667 - val_loss: 0.9508 - learning_rate: 1.0000e-04
Epoch 7/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 844ms/step - accuracy: 0.7080 - loss: 0.7419

46/46 ━━━━━━━━━━━━━━━━━━━━ 82s 916ms/step - accuracy: 0.7156 - loss: 0.7055 - val_accuracy: 0.7315 - val_loss: 0.8214 - learning_rate: 1.0000e-04
Epoch 8/30
33/46 ━━━━━━━━━━━━━━━━━━━━ 11s 885ms/step - accuracy: 0.7056 - loss: 0.6281

In [ ]:
loss, acc = model_cow.evaluate(test_data)
print("Final Accuracy:", acc)

In [ ]:
model_cow.save("/content/drive/MyDrive/cow_model_final.h5")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"/content/drive/MyDrive/ML_Models/cow_model.h5"

In [ ]:
model_cow.save("/content/drive/MyDrive/cow_model.h5")

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

In [ ]:
output = Dense(4, activation='softmax')(x)

In [ ]:
output = Dense(len(class_names), activation='softmax')(x)

In [ ]:
# 1. Upload image
from google.colab import files
uploaded = files.upload()

In [ ]:
from PIL import Image
import numpy as np

img_name = list(uploaded.keys())[0]

# Load the image using the variable img_name
img = Image.open(img_name)

# Convert image to RGB (remove alpha channel if present)
img = img.convert('RGB')

# Resize the image to the target size (224, 224) expected by the model
img = img.resize((224, 224))

# Convert the image to a NumPy array
img_array = np.array(img)

# Normalize pixel values to [0, 1]
img_array = img_array / 255.0

# Expand dimensions to add the batch size (e.g., (1, 224, 224, 3) for a single image)
img_array = np.expand_dims(img_array, axis=0)

# Make the prediction using the preprocessed img_array
pred = model_cow.predict(img_array)

class_idx = np.argmax(pred)
confidence = np.max(pred)

# Prepare results in a dictionary to match the original loop structure
result = {
    "Prediction": class_names[class_idx],
    "Confidence": f"{confidence * 100:.2f}%"
}

for k,v in result.items():
    print(k, ":", v)